In [31]:
import copy
import json
import os
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             classification_report)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)
print('Using:', DEVICE)
# Optional for quick developer checks; leave FAST_DEV_RUN unset for real training.
FAST_DEV_RUN = os.environ.get('FAST_DEV_RUN', '0') == '1'

Using: cpu


In [32]:
DATA_PATH = Path('adult.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Put adult.csv in the same folder as this script/notebook.')
df = pd.read_csv(DATA_PATH)
print('Original shape:', df.shape)
print(df.head(3))
print('Income labels:', df['income'].value_counts().to_dict())

Original shape: (32561, 15)
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   

        occupation   relationship   race     sex  capital.gain  capital.loss  \
0                ?  Not-in-family  White  Female             0          4356   
1  Exec-managerial  Not-in-family  White  Female             0          4356   
2                ?      Unmarried  Black  Female             0          4356   

   hours.per.week native.country income  
0              40  United-States  <=50K  
1              18  United-States  <=50K  
2              40  United-States  <=50K  
Income labels: {'<=50K': 24720, '>50K': 7841}


In [33]:
print('Column names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nNumeric summary:')
display(df.describe())
print('\nClass balance (%)')
display((df['income'].value_counts(normalize=True) * 100).round(2))

Column names: ['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']

Data types:
age               int64
workclass           str
fnlwgt            int64
education           str
education.num     int64
marital.status      str
occupation          str
relationship        str
race                str
sex                 str
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country      str
income              str
dtype: object

Numeric summary:


,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000



Class balance (%)


income
<=50K    75.92
>50K     24.08
Name: proportion, dtype: float64

In [34]:
text_columns = df.select_dtypes(include='object').columns
for col in text_columns:
    df[col] = df[col].str.strip()
df = df.replace('?', np.nan)
print('Missing values:', df.isna().sum()[df.isna().sum() > 0].to_dict())
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
# education.num is the numeric representation of education: avoid redundant inputs.
# fnlwgt is the census sampling weight, not an individual's income characteristic.
df = df.drop(columns=['education', 'fnlwgt'])
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
assert df['income'].notna().all(), 'Unexpected income label found!'
df['income'] = df['income'].astype(int)
print('Cleaned shape:', df.shape)
print('Target class counts:', df['income'].value_counts().to_dict())
# Save the cleaned table for inspection. Missing categories are filled LATER
# using the training split only, to prevent information leaking from test rows.
df.to_csv(OUT / 'cleaned_adult.csv', index=False)
print('Saved cleaned dataset:', OUT / 'cleaned_adult.csv')

Missing values: {'workclass': 1836, 'occupation': 1843, 'native.country': 583}
Duplicates: 24
Cleaned shape: (32537, 13)
Target class counts: {0: 24698, 1: 7839}
Saved cleaned dataset: outputs\cleaned_adult.csv


C:\Users\helen\AppData\Local\Temp\ipykernel_18208\1971187134.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include='object').columns


In [35]:
print('Missing values remaining after basic cleaning:')
print(df.isnull().sum().sort_values(ascending=False))

print('\nTotal rows after removing duplicates:', len(df))
print('Income 0 / 1 counts:')
print(df['income'].value_counts())

Missing values remaining after basic cleaning:
occupation        1843
workclass         1836
native.country     582
education.num        0
age                  0
marital.status       0
relationship         0
sex                  0
race                 0
capital.gain         0
capital.loss         0
hours.per.week       0
income               0
dtype: int64

Total rows after removing duplicates: 32537
Income 0 / 1 counts:
income
0    24698
1     7839
Name: count, dtype: int64


In [36]:
X = df.drop(columns=['income'])
y = df['income']
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85,
    random_state=SEED, stratify=y_train_val
)
print('Train, validation, test:', len(X_train), len(X_val), len(X_test))

Train, validation, test: 22775 4881 4881


In [37]:
# STEP 5: Impute, one-hot encode and scale using training rows ONLY.
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols),
])
Xtr = preprocessor.fit_transform(X_train).astype('float32')
Xva = preprocessor.transform(X_val).astype('float32')
Xte = preprocessor.transform(X_test).astype('float32')
ytr = y_train.to_numpy(dtype='float32')
yva = y_val.to_numpy(dtype='float32')
yte = y_test.to_numpy(dtype='float32')
print('Encoded feature count:', Xtr.shape[1])
joblib.dump(preprocessor, OUT / 'preprocessor.joblib')

Encoded feature count: 87


['outputs\\preprocessor.joblib']

In [38]:
# STEP 6: PyTorch DataLoaders (y is column of 0s and 1s).
BATCH_SIZE = 256
def make_loader(features, labels, shuffle=False):
    return DataLoader(
        TensorDataset(torch.tensor(features, dtype=torch.float32),
                      torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)),
        batch_size=BATCH_SIZE, shuffle=shuffle,
    )
train_loader = make_loader(Xtr, ytr, shuffle=True)
val_loader = make_loader(Xva, yva)
test_loader = make_loader(Xte, yte)

In [39]:
class IncomeMLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout=0.0, batch_norm=False):
        super().__init__()
        layers = []
        previous = input_size
        for width in hidden_sizes:
            layers.append(nn.Linear(previous, width))
            if batch_norm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = width
        layers.append(nn.Linear(previous, 1))  # one logit: income > $50k?
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [40]:
# STEP 8: Evaluate a model without using test data during model selection.
def evaluate(model, loader):
    model.eval()  # turns off Dropout and uses BatchNorm's running statistics
    actual, probabilities, losses = [], [], []
    criterion = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            losses.append((criterion(logits, yb).item(), len(xb)))
            actual.extend(yb.cpu().numpy().ravel().tolist())
            probabilities.extend(torch.sigmoid(logits).cpu().numpy().ravel().tolist())
    actual = np.array(actual, dtype=int)
    probabilities = np.array(probabilities)
    predicted = (probabilities >= 0.5).astype(int)
    return {
        'loss': sum(loss*n for loss, n in losses)/sum(n for _, n in losses),
        'accuracy': accuracy_score(actual, predicted),
        'precision': precision_score(actual, predicted, zero_division=0),
        'recall': recall_score(actual, predicted, zero_division=0),
        'f1': f1_score(actual, predicted, zero_division=0),
        'roc_auc': roc_auc_score(actual, probabilities),
    }

def get_probs(model, loader):
    model.eval()
    scores = []
    with torch.no_grad():
        for xb, _ in loader:
            scores.extend(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy().ravel().tolist())
    return np.array(scores)

In [41]:
# STEP 9: Define the three experiments (architectures, optimisers, learning rates).
EXPERIMENTS = [
    dict(name='Model_1_Simple_MLP', hidden=[32], dropout=0.0,
         batch_norm=False, optimizer='Adam', learning_rate=0.001,
         weight_decay=0.0, epochs=25),
    dict(name='Model_2_Deep_MLP', hidden=[128, 64, 32], dropout=0.0,
         batch_norm=False, optimizer='SGD', learning_rate=0.02,
         weight_decay=0.0, epochs=35),
    dict(name='Model_3_Regularised_MLP', hidden=[128, 64, 32], dropout=0.25,
         batch_norm=True, optimizer='AdamW', learning_rate=0.0008,
         weight_decay=0.0005, epochs=35),
]

In [42]:
# STEP 10: Start MLflow server FIRST in a separate terminal. Then run this script.
USE_MLFLOW = os.environ.get('SKIP_MLFLOW', '0') != '1'
if USE_MLFLOW:
    import mlflow
    mlflow.set_tracking_uri('http://127.0.0.1:5000')
    mlflow.set_experiment('Adult Census Income - Neural Networks')

class NoMLflow:
    """Offline test helper; the normal assignment run uses real MLflow."""
    def __enter__(self): return self
    def __exit__(self, *args): pass

def train_one(config):
    model = IncomeMLP(Xtr.shape[1], config['hidden'],
                      config['dropout'], config['batch_norm']).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    if config['optimizer'] == 'SGD':
        optim = torch.optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.9)
    elif config['optimizer'] == 'AdamW':
        optim = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'],
                                  weight_decay=config['weight_decay'])
    else:
        optim = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

    if FAST_DEV_RUN:
        config = {**config, 'epochs': 2}
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_val_loss = float('inf')
    best_weights = None
    best_epoch = 0
    patience, bad_epochs = 7, 0
    context = mlflow.start_run(run_name=config['name']) if USE_MLFLOW else NoMLflow()
    run_id = None
    with context:
        if USE_MLFLOW:
            run_id = mlflow.active_run().info.run_id
            mlflow.log_params({
                'hidden_layers': str(config['hidden']),
                'dropout': config['dropout'],
                'batch_norm': config['batch_norm'],
                'optimizer': config['optimizer'],
                'learning_rate': config['learning_rate'],
                'weight_decay': config['weight_decay'],
                'batch_size': BATCH_SIZE,
                'max_epochs': config['epochs'],
                'patience': patience,
                'features_after_encoding': Xtr.shape[1],
                'seed': SEED,
                'decision_threshold': 0.5,
            })
        for epoch in range(1, config['epochs'] + 1):
            model.train()
            running_loss, seen = 0.0, 0
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optim.zero_grad()
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optim.step()
                running_loss += loss.item()*len(xb)
                seen += len(xb)

            train_loss = running_loss/seen
            val = evaluate(model, val_loader)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val['loss'])
            history['val_f1'].append(val['f1'])
            if USE_MLFLOW:
                mlflow.log_metrics({'train_loss': train_loss,
                    'val_loss': val['loss'], 'val_accuracy': val['accuracy'],
                    'val_precision': val['precision'], 'val_recall': val['recall'],
                    'val_f1': val['f1'], 'val_roc_auc': val['roc_auc']}, step=epoch)
            if val['loss'] < best_val_loss - 0.0001:
                best_val_loss = val['loss']
                best_weights = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                bad_epochs = 0
            else:
                bad_epochs += 1
            if epoch == 1 or epoch % 5 == 0:
                print(f"{config['name']} | epoch {epoch:02} | train loss {train_loss:.4f} | val loss {val['loss']:.4f} | val F1 {val['f1']:.4f}")
            if bad_epochs >= patience:
                print('  Early stopping at epoch', epoch)
                break

        # Restore lowest-validation-loss weights (not last epoch weights).
        model.load_state_dict(best_weights)
        best_val = evaluate(model, val_loader)
        filename = OUT / f"{config['name']}.pt"
        torch.save(model.state_dict(), filename)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history['train_loss'], label='Training loss')
        ax.plot(history['val_loss'], label='Validation loss')
        ax.set(xlabel='Epoch', ylabel='Binary cross-entropy loss',
               title=config['name'] + ' – learning curves')
        ax.legend()
        fig.tight_layout()
        plot_path = OUT / f"{config['name']}_loss.png"
        fig.savefig(plot_path, dpi=150)
        plt.close(fig)
        if USE_MLFLOW:
            mlflow.log_metrics({f'best_val_{k}': float(v)
                                for k, v in best_val.items()})
            mlflow.log_metric('best_epoch', best_epoch)
            mlflow.log_artifact(str(filename), artifact_path='model_weights')
            mlflow.log_artifact(str(plot_path), artifact_path='plots')
            mlflow.log_artifact(str(OUT/'preprocessor.joblib'), artifact_path='preprocessing')
    return model, {'model': config['name'], 'mlflow_run_id': run_id,
                   'best_epoch': best_epoch,
                   **{f'val_{k}': v for k, v in best_val.items()}}

In [43]:
# STEP 11: Train all models and select the champion using validation metrics only.
trained_models = {}
model_results = []
for config in EXPERIMENTS:
    model, result = train_one(config)
    trained_models[result['model']] = model
    model_results.append(result)

comparison = pd.DataFrame(model_results).sort_values(
    ['val_f1', 'val_roc_auc'], ascending=[False, False]
).reset_index(drop=True)
comparison.to_csv(OUT / 'model_comparison.csv', index=False)

champion = comparison.iloc[0]
challenger = comparison.iloc[1]
champion_name = champion['model']
challenger_name = challenger['model']
champion_f1 = float(champion['val_f1'])
challenger_f1 = float(challenger['val_f1'])
champion_model = trained_models[champion_name]
challenger_model = trained_models[challenger_name]

print('Models ranked by validation F1 (ROC-AUC breaks ties):')
display(comparison)
print(f'Champion: {champion_name} (validation F1={champion_f1:.4f})')
print(f'Challenger: {challenger_name} (validation F1={challenger_f1:.4f})')

Model_1_Simple_MLP | epoch 01 | train loss 0.5348 | val loss 0.3955 | val F1 0.5199
Model_1_Simple_MLP | epoch 05 | train loss 0.3189 | val loss 0.3210 | val F1 0.6652
Model_1_Simple_MLP | epoch 10 | train loss 0.3109 | val loss 0.3143 | val F1 0.6556
Model_1_Simple_MLP | epoch 15 | train loss 0.3064 | val loss 0.3110 | val F1 0.6818
Model_1_Simple_MLP | epoch 20 | train loss 0.3031 | val loss 0.3089 | val F1 0.6729
Model_1_Simple_MLP | epoch 25 | train loss 0.3010 | val loss 0.3092 | val F1 0.6641
🏃 View run Model_1_Simple_MLP at: http://127.0.0.1:5000/#/experiments/1/runs/6a062666e87d400eb60b860403afa8df
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Model_2_Deep_MLP | epoch 01 | train loss 0.5571 | val loss 0.5058 | val F1 0.0000
Model_2_Deep_MLP | epoch 05 | train loss 0.3106 | val loss 0.3130 | val F1 0.6418
Model_2_Deep_MLP | epoch 10 | train loss 0.3041 | val loss 0.3099 | val F1 0.6941
Model_2_Deep_MLP | epoch 15 | train loss 0.2992 | val loss 0.3058 | val F1 0.687

,model,mlflow_run_id,best_epoch,val_loss,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,Model_3_Regularised_MLP,1c0f77b974ed4aaabb9c405411b9141c,10,0.307465,0.858840,0.743744,0.631803,0.683218,0.914716
1,Model_1_Simple_MLP,6a062666e87d400eb60b860403afa8df,22,0.308796,0.856382,0.746625,0.611395,0.672277,0.913324
2,Model_2_Deep_MLP,9df6714a173843fab2fa2e27b37a28df,18,0.305610,0.857816,0.764835,0.591837,0.667306,0.915533


Champion: Model_3_Regularised_MLP (validation F1=0.6832)
Challenger: Model_1_Simple_MLP (validation F1=0.6723)


In [44]:
# STEP 12: Save a clear Champion-vs-Challenger comparison and log it to MLflow.
# The bar plot and summary text are artifacts, useful for your presentation.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison['model'], comparison['val_f1'])
ax.set_ylabel('Validation F1-score (higher is better)')
ax.set_title('Champion vs Challenger vs Third Model')
ax.tick_params(axis='x', labelrotation=12)
for i, score in enumerate(comparison['val_f1']):
    ax.text(i, score + 0.006, f'{score:.4f}', ha='center')
ax.set_ylim(0, max(comparison['val_f1']) + 0.08)
fig.tight_layout()
comparison_plot = OUT / 'champion_challenger_comparison.png'
fig.savefig(comparison_plot, dpi=150)
plt.close(fig)

selection_file = OUT / 'champion_challenger_selection.txt'
selection_file.write_text(
    'CHAMPION: ' + champion_name + f' (validation F1={champion_f1:.4f})\n'
    'CHALLENGER: ' + challenger_name + f' (validation F1={challenger_f1:.4f})\n'
    'Selection rule: rank all three models by validation F1-score; '
    'use validation ROC-AUC to break ties.\n'
    'The test set was NOT used for choosing the roles.\n',
    encoding='utf-8',
)
if USE_MLFLOW:
    with mlflow.start_run(run_name='CHAMPION_vs_CHALLENGER'):
        mlflow.set_tag('model_role', 'Champion vs Challenger comparison')
        mlflow.log_param('selection_metric', 'validation F1-score')
        mlflow.log_param('champion_name', champion_name)
        mlflow.log_param('challenger_name', challenger_name)
        mlflow.log_metric('champion_validation_f1', champion_f1)
        mlflow.log_metric('challenger_validation_f1', challenger_f1)
        mlflow.log_metric('champion_minus_challenger_f1', champion_f1 - challenger_f1)
        mlflow.log_artifact(str(OUT / 'model_comparison.csv'), artifact_path='comparisons')
        mlflow.log_artifact(str(comparison_plot), artifact_path='comparisons')
        mlflow.log_artifact(str(selection_file), artifact_path='comparisons')

# STEP 12B: Save which network won, plus everything needed to reload its weights.
# The model weights alone are NOT enough: predictions need the matching preprocessor
# and the exact architecture (hidden layer sizes, Dropout, BatchNorm and input size).
model_configurations = {c['name']: c for c in EXPERIMENTS}
champion_architecture = model_configurations[champion_name]
champion_info = {
    'champion_name': champion_name,
    'challenger_name': challenger_name,
    'selection_metric': 'validation F1-score',
    'validation_champion_f1': champion_f1,
    'validation_challenger_f1': challenger_f1,
    'input_size': int(Xtr.shape[1]),
    'feature_columns': list(X_train.columns),
    'hidden': champion_architecture['hidden'],
    'dropout': champion_architecture['dropout'],
    'batch_norm': champion_architecture['batch_norm'],
    'threshold': 0.5,
    'champion_weights': f'{champion_name}.pt',
    'preprocessor_file': 'preprocessor.joblib',
}
(OUT / 'champion_info.json').write_text(
    json.dumps(champion_info, indent=2), encoding='utf-8'
)
# These two CSV rows can be used by predict.py to demonstrate new predictions.
# They do not contain the 'income' answer column.
example_lower = X_test.loc[y_test[y_test == 0].index].iloc[[0]]
example_higher = X_test.loc[y_test[y_test == 1].index].iloc[[0]]
sample_people = pd.concat([example_lower, example_higher], ignore_index=True)
sample_people.to_csv('sample_people.csv', index=False)
print('Saved prediction metadata: outputs/champion_info.json')
print('Saved new-person example input: sample_people.csv')



🏃 View run CHAMPION_vs_CHALLENGER at: http://127.0.0.1:5000/#/experiments/1/runs/531f9deeb4ab46fc9abee0d108017b61
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Saved prediction metadata: outputs/champion_info.json
Saved new-person example input: sample_people.csv


In [45]:
# STEP 13: Only now use the held-out TEST dataset, only for the champion.
# Never use the test results to choose between Champion and Challenger.
test = evaluate(champion_model, test_loader)
print('\nCHAMPION FINAL TEST RESULTS (unseen data):')
for key, val in test.items():
    print(f' {key}: {val:.4f}')
probs = get_probs(champion_model, test_loader)
predictions = (probs >= 0.5).astype(int)
# Save the predictions for a detailed model evaluation table.
test_prediction_table = pd.DataFrame({
    'actual_income_class': yte.astype(int),
    'predicted_income_class': predictions,
    'probability_above_50k': np.round(probs, 6),
})
test_prediction_table.to_csv(OUT / 'test_predictions.csv', index=False)
print('Saved test predictions: outputs/test_predictions.csv')
print('\nTest classification report:\n', classification_report(
    yte.astype(int), predictions, target_names=['<=50K', '>50K'], zero_division=0))
ConfusionMatrixDisplay.from_predictions(yte.astype(int), predictions,
    display_labels=['<=50K', '>50K'], cmap='Blues')
plt.title('Champion test confusion matrix: ' + champion_name)
plt.tight_layout()
plt.savefig(OUT / 'champion_confusion_matrix.png', dpi=150)
plt.close()
with open(OUT / 'champion_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Champion: ' + champion_name + '\n')
    f.write('Challenger: ' + challenger_name + '\n')
    f.write('Selection based on VALIDATION F1, not test set.\n')
    for key, val in test.items():
        f.write(f'{key}: {val:.4f}\n')




CHAMPION FINAL TEST RESULTS (unseen data):
 loss: 0.3193
 accuracy: 0.8545
 precision: 0.7344
 recall: 0.6207
 f1: 0.6728
 roc_auc: 0.9070
Saved test predictions: outputs/test_predictions.csv

Test classification report:
               precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      3705
        >50K       0.73      0.62      0.67      1176

    accuracy                           0.85      4881
   macro avg       0.81      0.77      0.79      4881
weighted avg       0.85      0.85      0.85      4881



In [46]:
# STEP 14: Log the Champion's final test evaluation in a separate MLflow run.
if USE_MLFLOW:
    with mlflow.start_run(run_name='CHAMPION_FINAL_TEST_' + champion_name):
        mlflow.set_tag('model_role', 'Champion final test')
        mlflow.set_tag('purpose', 'Final test only after validation model selection')
        mlflow.log_param('champion_model', champion_name)
        mlflow.log_param('challenger_model', challenger_name)
        mlflow.log_param('selection_metric', 'validation F1-score')
        mlflow.log_metrics({f'test_{k}': float(v) for k, v in test.items()})
        mlflow.log_artifact(str(OUT/'champion_confusion_matrix.png'), artifact_path='plots')
        mlflow.log_artifact(str(OUT/'model_comparison.csv'), artifact_path='comparisons')
        mlflow.log_artifact(str(OUT/'champion_summary.txt'), artifact_path='results')
        mlflow.log_artifact(str(OUT/'champion_info.json'), artifact_path='inference')
        mlflow.log_artifact(str(OUT/'test_predictions.csv'), artifact_path='results')


🏃 View run CHAMPION_FINAL_TEST_Model_3_Regularised_MLP at: http://127.0.0.1:5000/#/experiments/1/runs/a71971e2dbbc495eae92933256e837f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [47]:
if USE_MLFLOW:
    runs = mlflow.search_runs(
        experiment_names=['Adult Census Income - Neural Networks']
    )
    columns = ['tags.mlflow.runName', 'tags.model_role',
               'metrics.best_val_f1', 'metrics.best_val_roc_auc']
    available = [column for column in columns if column in runs.columns]
    print('MLflow runs:', len(runs))
    display(runs[available].fillna('—'))
else:
    print('MLflow was disabled in this run. Start MLflow and run again normally.')

MLflow runs: 5


,tags.mlflow.runName,tags.model_role,metrics.best_val_f1,metrics.best_val_roc_auc
0,CHAMPION_FINAL_TEST_Model_3_Regularised_MLP,Champion final test,—,—
1,CHAMPION_vs_CHALLENGER,Champion vs Challenger comparison,—,—
2,Model_3_Regularised_MLP,—,0.683218,0.914716
3,Model_2_Deep_MLP,—,0.667306,0.915533
4,Model_1_Simple_MLP,—,0.672277,0.913324


In [48]:
# Predict 1 person's income inside the notebook
example_row = X_test.iloc[[0]]
example_scaled = preprocessor.transform(example_row).astype('float32')
champion_model.eval()
with torch.no_grad():
    logit = champion_model(torch.tensor(example_scaled).to(DEVICE))
    probability = torch.sigmoid(logit).cpu().item()
label = '>50K' if probability >= 0.5 else '<=50K'
print('Predicted income:', label)
print('Probability of >50K:', round(probability, 4))
print('Known test label (for checking only):', '>50K' if y_test.iloc[0] == 1 else '<=50K')


Predicted income: <=50K
Probability of >50K: 0.0043
Known test label (for checking only): <=50K


In [52]:
# STEP 17: Predict from CSV using files saved by training (no retraining needed)
from predict import predict_csv
predicted_people = predict_csv('adult.csv', 'outputs/new_predictions.csv')
display(predicted_people[['probability_above_50k', 'predicted_class', 'predicted_income']])

,probability_above_50k,predicted_class,predicted_income
0,0.166397,0,<=50K
1,0.189938,0,<=50K
2,0.402976,0,<=50K
3,0.042404,0,<=50K
4,0.400995,0,<=50K
...,...,...,...
32556,0.013789,0,<=50K
32557,0.535905,1,>50K
32558,0.250094,0,<=50K
32559,0.031165,0,<=50K
